# Categorical features: one-hot encoding

<a target="_blank" href="https://colab.research.google.com/github/changyaochen/MECE4520/blob/master/site/regression/05-one-hot-encoding.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

So far, every feature in our regression models has been numerical. This notebook adds `campaign_year`, the year in which a set of gas-turbine measurements was collected. The goal is not to interpret year as an operating lever; it is to practice including a **categorical feature** in a simple model.

We fit the same model in two ways:

1. `statsmodels` formula syntax, which recognizes a categorical feature through `C(...)`.
2. A scikit-learn pipeline, where we explicitly specify one-hot encoding.

By the end, the two fits should make the same predictions.

## 1. Load the measurements

Each row is an hourly aggregate from the course gas-turbine emissions dataset. **AT** is ambient temperature (°C); **NOX** is nitrogen-oxides concentration in the exhaust (mg/m³). `campaign_year` identifies the measurement campaign.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Use the local course CSV when available; download it in Colab otherwise.
DATA_URL = (
    "https://raw.githubusercontent.com/changyaochen/MECE4520/master/"
    "site/data/gas-turbine-course.csv"
)
data_path = Path("../data/gas-turbine-course.csv")
if not data_path.exists():
    data_path = Path("gas-turbine-course.csv")
    if not data_path.exists():
        urlretrieve(DATA_URL, data_path)

data = pd.read_csv(data_path)
regression_data = data[["campaign_year", "AT", "NOX"]].dropna().copy()

# Sample three operating hours from each campaign, rather than showing only 2011.
regression_data.groupby("campaign_year", group_keys=False).sample(n=3, random_state=4520).sort_values("campaign_year")

,campaign_year,AT,NOX
2682,2011,14.7380,69.508
5959,2011,14.5360,60.553
5189,2011,28.1130,62.574
13042,2012,20.2000,67.404
9136,2012,13.4630,71.540
13892,2012,13.1720,84.821
15630,2013,6.2945,67.589
19862,2013,29.5780,57.711
16274,2013,11.9720,77.311
22208,2014,9.3223,69.967


The five campaign years are labels, not measurements on a continuous scale. Treating them as the numbers 2011, 2012, ..., 2015 would force the model to assume that each one-year step has the same effect on NOx. We do not have a physical reason to impose that assumption.

Instead, we create an indicator (a 0/1 column) for every year **except one reference year**. We deliberately leave out the 2011 indicator; its effect is represented by the intercept. The population model is

$$
\mathrm{NOX}_i = \beta_0 + \beta_{\mathrm{AT}}\,\mathrm{AT}_i
+ \gamma_{2012}D_{i,2012} + \gamma_{2013}D_{i,2013}
+ \gamma_{2014}D_{i,2014} + \gamma_{2015}D_{i,2015} + \varepsilon_i.
$$

For example, $D_{i,2013}=1$ when row $i$ belongs to the 2013 campaign and 0 otherwise. There is no $D_{i,2011}$ column: 2011 is the baseline. The intercept is the modeled baseline for 2011, and each $\gamma$ is a difference from that baseline at the same ambient temperature.

## 2. `statsmodels`: mark the feature as categorical

The `statsmodels` formula interface can construct the indicator columns for us. In the formula, `C(campaign_year)` means “treat `campaign_year` as categorical,” rather than as a numeric variable.

In [2]:
# C(...) creates treatment-coded indicator columns internally.
statsmodels_model = smf.ols(
    "NOX ~ AT + C(campaign_year)",
    data=regression_data,
).fit()

statsmodels_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    NOX   R-squared:                       0.455
Model:                            OLS   Adj. R-squared:                  0.455
Method:                 Least Squares   F-statistic:                     6131.
Date:                Mon, 21 Sep 2026   Prob (F-statistic):               0.00
Time:                        23:20:00   Log-Likelihood:            -1.3126e+05
No. Observations:               36733   AIC:                         2.625e+05
Df Residuals:                   36727   BIC:                         2.626e+05
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
============================================================================================
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                   82.6686      0.144    573.606      0.000      82.386      82.951
C(campaign_year)[T.2012]     2.3476      0.141     16.667      0.000       2.071       2.624
C(campaign_year)[T.2013]     2.8650      0.143     20.041      0.000       2.585       3.145
C(campaign_year)[T.2014]    -6.5341      0.143    -45.676      0.000      -6.814      -6.254
C(campaign_year)[T.2015]    -7.5852      0.142    -53.500      0.000      -7.863      -7.307
AT                          -0.8820      0.006   -145.649      0.000      -0.894      -0.870
==============================================================================
Omnibus:                     8297.758   Durbin-Watson:                   0.386
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            29067.008
Skew:                           1.119   Prob(JB):                         0.00
Kurtosis:                       6.740   Cond. No.                         105.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

`C(campaign_year)[T.2012]`, for example, is the estimated change for the 2012 campaign relative to 2011 while holding AT fixed. There is no coefficient for 2011 because it is the reference category.

Why omit one indicator? If we included an intercept and all five year indicators, one column would be exactly determined by the others: every row belongs to exactly one year. Removing one indicator avoids this exact redundancy, often called the **dummy-variable trap**.

In [3]:
# Display just the fitted coefficients in a compact form.
statsmodels_model.params.rename("coefficient").to_frame().round(4)

,coefficient
Intercept,82.6686
C(campaign_year)[T.2012],2.3476
C(campaign_year)[T.2013],2.8650
C(campaign_year)[T.2014],-6.5341
C(campaign_year)[T.2015],-7.5852
AT,-0.8820


## 3. `pandas`: create the indicator columns yourself

`pandas.get_dummies` is a useful middle ground between the two approaches. It makes the one-hot columns visible in a DataFrame, then we pass that fully numerical table to the same matrix-based `statsmodels` interface used in earlier regression notebooks.

Use this approach here to see the mechanics. For a practical scikit-learn workflow, prefer the pipeline in the next section: it applies the same encoding automatically when new data arrive.

As before, `drop_first=True` leaves out 2011, the reference year.

In [4]:
# Turn the categorical year column into four visible 0/1 indicator columns.
dummy_data = pd.get_dummies(
    regression_data,
    columns=["campaign_year"],
    drop_first=True,
    dtype=float,
)
dummy_data.head()

,AT,NOX,campaign_year_2012,campaign_year_2013,campaign_year_2014,campaign_year_2015
0,4.5878,81.952,0.0,0.0,0.0,0.0
1,4.2932,82.377,0.0,0.0,0.0,0.0
2,3.9045,83.776,0.0,0.0,0.0,0.0
3,3.7436,82.505,0.0,0.0,0.0,0.0
4,3.7516,82.028,0.0,0.0,0.0,0.0


In [5]:
# OLS now receives only numerical columns, including the four year indicators.
dummy_features = [column for column in dummy_data.columns if column != "NOX"]
dummy_design = sm.add_constant(dummy_data[dummy_features])
pandas_model = sm.OLS(dummy_data["NOX"], dummy_design).fit()
pandas_model.params.rename("coefficient").to_frame().round(4)

,coefficient
const,82.6686
AT,-0.8820
campaign_year_2012,2.3476
campaign_year_2013,2.8650
campaign_year_2014,-6.5341
campaign_year_2015,-7.5852


## 4. Scikit-learn: specify the encoding explicitly

[Scikit-learn](https://scikit-learn.org/) is a widely used Python library for machine learning. Its workflow separates **preprocessing** from an **estimator**: an object transforms the input features, then a model object learns from the transformed numerical matrix. A `Pipeline` connects those steps so exactly the same transformations are used whenever we fit or predict.

Scikit-learn estimators work with a numerical feature matrix, so we tell the preprocessing step exactly how to transform each column. `ColumnTransformer` passes AT through unchanged and applies `OneHotEncoder` to `campaign_year`.

`drop="first"` removes the first sorted year (2011), giving this model the same reference category as the `statsmodels` fit.

In [6]:
features = regression_data[["AT", "campaign_year"]]
target = regression_data["NOX"]

# Define the numerical and categorical transformations.
preprocessor = ColumnTransformer(
    transformers=[
        ("temperature", "passthrough", ["AT"]),
        (
            "year",
            OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
            ["campaign_year"],
        ),
    ]
)

# Fit the encoder and the linear regression together.
sklearn_model = Pipeline(
    steps=[
        ("encode", preprocessor),
        ("regression", LinearRegression()),
    ]
)
sklearn_model.fit(features, target)

,steps,"[('encode', ...), ('regression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('temperature', ...), ('year', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [7]:
# Retrieve the transformed feature names and their fitted coefficients.
feature_names = sklearn_model.named_steps["encode"].get_feature_names_out()
coefficient_table = pd.DataFrame(
    {"coefficient": sklearn_model.named_steps["regression"].coef_},
    index=feature_names,
)
coefficient_table.loc["Intercept"] = sklearn_model.named_steps["regression"].intercept_
coefficient_table.round(4)

,coefficient
temperature__AT,-0.8820
year__campaign_year_2012,2.3476
year__campaign_year_2013,2.8650
year__campaign_year_2014,-6.5341
year__campaign_year_2015,-7.5852
Intercept,82.6686


The names differ slightly between packages, but the coefficients have the same interpretation. Plain scikit-learn focuses on fitting and prediction; unlike the `statsmodels` summary, it does not report coefficient standard errors, confidence intervals, or p-values.

## 5. Confirm that all three models agree

All three approaches fit ordinary least squares with the same features, reference year, and rows. Their numerical predictions should therefore agree up to floating-point rounding.

In [8]:
statsmodels_predictions = statsmodels_model.predict(regression_data)
pandas_predictions = pandas_model.predict(dummy_design)
sklearn_predictions = sklearn_model.predict(features)

prediction_comparison = pd.DataFrame(
    {
        "statsmodels formula": statsmodels_predictions,
        "pandas dummies": pandas_predictions,
        "scikit-learn": sklearn_predictions,
    }
)
largest_differences = prediction_comparison.sub(
    prediction_comparison["statsmodels formula"], axis=0
).abs().max()
largest_differences.rename("largest difference from formula fit")

np.testing.assert_allclose(statsmodels_predictions, pandas_predictions)
np.testing.assert_allclose(statsmodels_predictions, sklearn_predictions)

## Takeaways

- Use one-hot encoding when a feature represents categories without a meaningful numeric spacing.
- With an intercept, retain one reference category and encode the others as differences from it.
- `pandas.get_dummies` is useful for inspecting the mechanics. In practice, use a scikit-learn preprocessing pipeline so encoding is applied consistently to new data.
- The campaign-year coefficients describe differences among these measurement campaigns. They do **not** show that changing the calendar year would cause NOx to change.

[Return to Regression](index.qmd)